# Riesgo de fuga a otro comercializador

Cuarto producto del proyecto: ¿qué clientes tienen más probabilidad de irse a otro
comercializador en los próximos 6 meses?

**De dónde salen los ejemplos.** De la carpeta `00_otros_comercializadores`: el archivo
de la empresa con los usuarios que ya están atendidos por otros comercializadores
(registro mensual de peajes). Se cruza con la historia TC2 y, para los que tienen historia,
se ubica el **mes de salida**: el más temprano entre su primer mes en ese archivo, su primer
mes de consumo cero sostenido en TC2 (así se ve una salida en la facturación: el cliente sigue
apareciendo con 0 kWh) y el mes siguiente a su última fila en TC2. Si algún mes el TC2 trae
el ciclo 97 (OTROS COMERCIALIZADORES), también cuenta como salida, automáticamente.

**A quién se le calcula el riesgo.** Población automática: las clases de servicio que aparecen
entre los que se fueron (hoy comercial, industrial, oficial, acueductos y no regulados), sin
alumbrado público, provisionales ni ciclos internos de EBSA. Si el archivo trae algún día un
residencial, la población se amplía sola.

**Cómo se usa.** `EBSA_MODO=reentrenar` entrena y evalúa por cortes en el tiempo (nunca con el
futuro) y guarda el modelo; `EBSA_MODO=aplicar` usa el modelo guardado y solo puntúa el corte
nuevo. Todas las salidas quedan en `10_riesgo_fuga`, con copia por corte en `historial/` y un
seguimiento de cuántos de los señalados efectivamente se fueron.

In [ ]:
# ============================================================
# 1. LIBRERÍAS, RUTAS Y CONFIGURACIÓN
# ============================================================
from pathlib import Path
import os
import re
import gc
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

from utilidades_glosario import (
    enriquecer_glosario, grupo_desde_perfil, grupo_desde_mediana, nombre_zona, nombre_clase,
    CICLOS_INTERNOS, CICLOS_SIN_GESTION, CICLO_OTROS_COMERCIALIZADORES, CICLO_NO_REGULADOS, GRUPOS_CONSUMO_ORDEN,
)

BASE_DIR = Path(os.environ.get("EBSA_DATOS", r"C:\Users\Home\Documents\Datos_Ebsa"))
MODO = os.environ.get("EBSA_MODO", "reentrenar").strip().lower()
if MODO not in ("reentrenar", "aplicar"):
    raise ValueError(f"EBSA_MODO debe ser 'reentrenar' o 'aplicar', no '{MODO}'")

OTROS_DIR = BASE_DIR / "00_otros_comercializadores"
PROCESADO_DIR = BASE_DIR / "01_historico_procesado"
PREPROC_DIR = BASE_DIR / "03_serie_modelado"
MODELO_DIR = BASE_DIR / "04_pronostico" / "modelo_final"
CAIDA_DIR = BASE_DIR / "06_estudio_caida"

FUGA_DIR = BASE_DIR / "10_riesgo_fuga"
HISTORIAL_DIR = FUGA_DIR / "historial"
FUGA_DIR.mkdir(parents=True, exist_ok=True)
HISTORIAL_DIR.mkdir(parents=True, exist_ok=True)

RUTA_SERIE = PREPROC_DIR / "serie_mensual_modelado_preprocesada.parquet"
RUTA_PERFILES = MODELO_DIR / "perfiles_consumidores_corte_final.parquet"
RUTA_CAIDA = CAIDA_DIR / "estudio_caida_consumo.parquet"

RUTA_MODELO = FUGA_DIR / "modelo_riesgo_fuga.joblib"
RUTA_SCORES = FUGA_DIR / "riesgo_fuga_clientes.csv"
RUTA_GERENCIAL = FUGA_DIR / "lista_riesgo_fuga_gerencial.csv"
RUTA_POR_ZONA = FUGA_DIR / "lista_riesgo_fuga_por_zona.csv"
RUTA_YA_FUERA = FUGA_DIR / "clientes_con_otro_comercializador.csv"
RUTA_VIGILANCIA = FUGA_DIR / "vigilancia_mercado_no_regulado.csv"
RUTA_PERFIL_IDOS = FUGA_DIR / "perfil_clientes_que_se_fueron.csv"
RUTA_RESUMEN_ZONA = FUGA_DIR / "resumen_riesgo_fuga_por_zona.csv"
RUTA_RESUMEN_GRUPO = FUGA_DIR / "resumen_riesgo_fuga_por_grupo.csv"
RUTA_METRICAS = FUGA_DIR / "metricas_modelo_fuga.csv"
RUTA_IMPORTANCIA = FUGA_DIR / "importancia_variables_fuga.csv"
RUTA_SEGUIMIENTO = FUGA_DIR / "seguimiento_riesgo_fuga.csv"
RUTA_ETIQUETAS = FUGA_DIR / "etiquetas_salida_por_niu.csv"

# Corte máximo (solo para simulaciones mes a mes con pipeline --corte-max): nada posterior
# a este mes se usa, ni del histórico ni del archivo de otros comercializadores.
CORTE_MAX = os.environ.get("EBSA_CORTE_MAX", "").strip()
CORTE_MAX = pd.Timestamp(CORTE_MAX + "-01").to_period("M").to_timestamp() if CORTE_MAX else None
if CORTE_MAX is not None:
    print(f"CORTE MÁXIMO (simulación): se ignora todo lo posterior a {CORTE_MAX:%Y-%m}")

SEED = 42
HORIZONTE_MESES = 6            # "se va en los próximos 6 meses"
MIN_MESES_HISTORIA = 6         # meses válidos en los últimos 12 para puntuar / entrenar
MESES_CERO_SOSTENIDO = 3       # racha de ceros que se interpreta como salida (en clientes del archivo)
MIN_POSITIVOS_MODELO = 20      # con menos ejemplos no se entrena un modelo: se usa similitud
CORTES_VALIDACION = 6          # últimos cortes con ventana completa que se usan para evaluar
MAX_FILAS_NEGATIVAS = 300_000  # muestreo de negativos para entrenar (los positivos van todos)
MESES_MAX_SIN_REENTRENAR = 6
CLASES_EXCLUIDAS_SIEMPRE = {"AP", "PR"}       # alumbrado público y provisionales
CLASES_POR_DEFECTO = {"CR", "ID", "OF", "AA", "IR"}   # solo si no hay archivo de otros comercializadores
UMBRAL_NO_REGULADO_KWH = 55_000.0            # candidato a mercado no regulado (≈ 55 MWh-mes)
N_ENSAYOS_OPTUNA = 30          # búsqueda de hiperparámetros (modo reentrenar), con semilla: misma serie -> mismo resultado
MULT_ALTO = 5.0                # nivel ALTO : probabilidad >= 5 x tasa base, o dentro del 1% más alto
MULT_MEDIO = 2.0               # nivel MEDIO: probabilidad >= 2 x tasa base, o dentro del 5% más alto
PCT_TOP_ALTO = 0.01            # (así siempre hay una lista corta para trabajar, aunque el riesgo general sea bajo)
PCT_TOP_MEDIO = 0.05
MIN_TOP_ALTO = 10

print(f"MODO DE EJECUCIÓN: {MODO.upper()}")
print("Datos             :", BASE_DIR)
print("Otros comerciali. :", OTROS_DIR)
print("Salidas           :", FUGA_DIR)


## Archivo de usuarios atendidos por otros comercializadores

In [ ]:
# ============================================================
# 2. LEER EL ARCHIVO DE OTROS COMERCIALIZADORES (todos los xlsx de la carpeta)
# ============================================================
# El archivo de la empresa no se modifica. Se aceptan varios archivos (uno por
# entrega); las filas repetidas NIU-mes se dejan una sola vez.
# ============================================================

def normalizar_niu(s):
    s = s.astype("string").str.strip().str.replace(r"\D", "", regex=True).str.lstrip("0")
    return s.replace("", pd.NA)

def leer_otros(ruta):
    d = pd.read_excel(ruta, sheet_name=0)
    d.columns = [re.sub(r"\s+", " ", str(c)).strip().upper() for c in d.columns]
    faltan = [c for c in ["AÑO", "MES", "NIU"] if c not in d.columns]
    if faltan:
        raise ValueError(f"{ruta.name}: faltan columnas {faltan}. Se esperan AÑO, MES, NIU (y opcionalmente "
                         "COMERCIALIZADOR, MUNICIPIO, ZONA, NIVEL DE TENSION, CONSUMO, USUARIO).")
    d["periodo"] = pd.to_datetime(d["AÑO"].astype(int).astype(str) + "-" + d["MES"].astype(int).astype(str).str.zfill(2) + "-01")
    d["NIU"] = normalizar_niu(d["NIU"])
    d = d[d["NIU"].notna()]
    for c in ["COMERCIALIZADOR", "MUNICIPIO", "USUARIO"]:
        if c not in d.columns:
            d[c] = pd.NA
    d["ZONA_ARCHIVO"] = pd.to_numeric(d["ZONA"], errors="coerce") if "ZONA" in d.columns else np.nan
    d["NIVEL_TENSION"] = pd.to_numeric(d["NIVEL DE TENSION"], errors="coerce") if "NIVEL DE TENSION" in d.columns else np.nan
    d["CONSUMO_OTRO"] = pd.to_numeric(d["CONSUMO"], errors="coerce") if "CONSUMO" in d.columns else np.nan
    d["archivo"] = ruta.name
    return d[["NIU", "periodo", "COMERCIALIZADOR", "MUNICIPIO", "USUARIO", "ZONA_ARCHIVO", "NIVEL_TENSION", "CONSUMO_OTRO", "archivo"]]

archivos_otros = sorted(OTROS_DIR.glob("*.xls*")) if OTROS_DIR.exists() else []
if archivos_otros:
    otros = pd.concat([leer_otros(a) for a in archivos_otros], ignore_index=True)
    otros = otros.sort_values(["NIU", "periodo", "archivo"]).drop_duplicates(["NIU", "periodo"], keep="last")
    if CORTE_MAX is not None:
        otros = otros[otros["periodo"] <= CORTE_MAX]
    HAY_ARCHIVO_OTROS = True
    print("ARCHIVO(S) DE OTROS COMERCIALIZADORES")
    print("-" * 70)
    print("Archivos:", ", ".join(a.name for a in archivos_otros))
    print(f"Filas: {len(otros):,} | NIU: {otros['NIU'].nunique():,} | "
          f"meses: {otros['periodo'].min():%Y-%m} -> {otros['periodo'].max():%Y-%m}")
    display(otros.groupby("COMERCIALIZADOR")["NIU"].nunique().sort_values(ascending=False).head(12)
            .rename("n_NIU").to_frame())
else:
    otros = pd.DataFrame(columns=["NIU", "periodo", "COMERCIALIZADOR", "MUNICIPIO", "USUARIO", "ZONA_ARCHIVO",
                                  "NIVEL_TENSION", "CONSUMO_OTRO", "archivo"])
    HAY_ARCHIVO_OTROS = False
    print("⚠ No hay archivos en", OTROS_DIR)
    print("  Sin ejemplos de clientes que se fueron, el riesgo se calcula solo con el ciclo 97 (si existe)")
    print("  y con la población por defecto:", sorted(CLASES_POR_DEFECTO))

# Resumen por NIU del archivo de otros
if HAY_ARCHIVO_OTROS:
    por_niu_otros = (otros.sort_values("periodo").groupby("NIU")
                     .agg(primer_mes_otro=("periodo", "min"), ultimo_mes_otro=("periodo", "max"),
                          meses_en_archivo=("periodo", "nunique"), comercializador=("COMERCIALIZADOR", "last"),
                          municipio=("MUNICIPIO", "last"), usuario=("USUARIO", "last"),
                          zona_archivo=("ZONA_ARCHIVO", "last"), nivel_tension=("NIVEL_TENSION", "last"),
                          consumo_prom_otro_kwh=("CONSUMO_OTRO", "mean")))
    ULTIMO_MES_ARCHIVO_OTROS = otros["periodo"].max()
    PRIMER_MES_ARCHIVO_OTROS = otros["periodo"].min()
else:
    por_niu_otros = pd.DataFrame(columns=["primer_mes_otro", "ultimo_mes_otro", "meses_en_archivo", "comercializador",
                                          "municipio", "usuario", "zona_archivo", "nivel_tension", "consumo_prom_otro_kwh"])
    por_niu_otros.index.name = "NIU"
    ULTIMO_MES_ARCHIVO_OTROS = pd.NaT
    PRIMER_MES_ARCHIVO_OTROS = pd.NaT


## Historia TC2: atributos por cliente y serie de consumo

In [ ]:
# ============================================================
# 3. ATRIBUTOS POR NIU DESDE EL HISTÓRICO (leyendo archivo por archivo)
# ============================================================
# Por NIU: primera y última fila en TC2, último ciclo / clase / estrato /
# tipo de medidor / tipo de lectura / promedio semestral / valor facturado,
# última tarifa real (> 0) y primer mes en ciclo 97 si lo hubo.
# ============================================================

patron = re.compile(r"^historico_(20\d{2})\.parquet$", re.IGNORECASE)
archivos_hist = sorted([p for p in PROCESADO_DIR.glob("historico_*.parquet") if patron.match(p.name)],
                       key=lambda p: int(patron.match(p.name).group(1)))
if not archivos_hist:
    raise FileNotFoundError(f"No hay historico_YYYY.parquet en {PROCESADO_DIR}")

COLS_H = ["NIU", "periodo", "ciclo", "clase_servicio", "estrato", "tipo_medidor", "tipo_lectura",
          "tarifa_aplicada_kwh", "consumo_promedio_semestral_kwh", "consumo_kwh_raw", "valor_facturado_consumo"]

ultimas, primeras, tarifas, c97 = [], [], [], []
for ruta in archivos_hist:
    esquema = pq.ParquetFile(ruta).schema_arrow.names
    cols = [c for c in COLS_H if c in esquema]
    h = pd.read_parquet(ruta, columns=cols, engine="pyarrow")
    h["NIU"] = normalizar_niu(h["NIU"])
    h = h[h["NIU"].notna()]
    h["periodo"] = pd.to_datetime(h["periodo"], errors="coerce")
    if CORTE_MAX is not None:
        h = h[h["periodo"] <= CORTE_MAX]
        if len(h) == 0:
            continue
    h["ciclo"] = pd.to_numeric(h["ciclo"], errors="coerce") if "ciclo" in h else np.nan
    if "clase_servicio" in h:
        h["clase_servicio"] = h["clase_servicio"].astype("string").str.strip().str.upper()
    h = h.sort_values(["NIU", "periodo"])
    ultimas.append(h.groupby("NIU").last().assign(_anio=int(patron.match(ruta.name).group(1))))
    primeras.append(h.groupby("NIU")["periodo"].min().rename("primer_mes_tc2"))
    if "tarifa_aplicada_kwh" in h:
        t = h[pd.to_numeric(h["tarifa_aplicada_kwh"], errors="coerce") > 0]
        tarifas.append(t.groupby("NIU")[["periodo", "tarifa_aplicada_kwh"]].last())
    m97 = h[h["ciclo"].eq(CICLO_OTROS_COMERCIALIZADORES)]
    if len(m97):
        c97.append(m97.groupby("NIU")["periodo"].min().rename("primer_mes_ciclo97"))
    del h
    gc.collect()

ultima_fila = pd.concat(ultimas).sort_values("periodo")
ultima_fila = ultima_fila.groupby(level=0).last()
atrib = ultima_fila.rename(columns={"periodo": "ultimo_mes_tc2", "consumo_kwh_raw": "consumo_ultima_fila_kwh"})
atrib["primer_mes_tc2"] = pd.concat(primeras).groupby(level=0).min()
if tarifas:
    tar = pd.concat(tarifas).sort_values("periodo").groupby(level=0).last()
    atrib["tarifa_kwh"] = tar["tarifa_aplicada_kwh"]
    atrib["periodo_tarifa"] = tar["periodo"]
else:
    atrib["tarifa_kwh"] = np.nan
    atrib["periodo_tarifa"] = pd.NaT
atrib["primer_mes_ciclo97"] = pd.concat(c97).groupby(level=0).min() if c97 else pd.NaT
atrib = atrib.drop(columns=["_anio", "tarifa_aplicada_kwh"], errors="ignore")
for c in ["estrato", "tipo_medidor", "tipo_lectura", "consumo_promedio_semestral_kwh", "valor_facturado_consumo"]:
    if c not in atrib.columns:
        atrib[c] = np.nan
atrib["estrato"] = pd.to_numeric(atrib["estrato"], errors="coerce")
ULTIMO_MES_TC2 = atrib["ultimo_mes_tc2"].max()

print("ATRIBUTOS POR NIU (HISTÓRICO TC2)")
print("-" * 70)
print(f"NIU: {len(atrib):,} | último mes del histórico: {ULTIMO_MES_TC2:%Y-%m}")
print(f"NIU con ciclo 97 (OTROS COMERCIALIZADORES) en TC2: {atrib['primer_mes_ciclo97'].notna().sum():,}")
display(atrib["clase_servicio"].value_counts().head(12).rename("n_NIU").to_frame().T)
del ultimas, primeras, tarifas, c97, ultima_fila
_ = gc.collect()


In [ ]:
# ============================================================
# 4. FECHA DE CORTE Y SERIE MENSUAL DE LA POBLACIÓN
# ============================================================
# La fecha de corte es la misma del estudio de caída (último mes consolidado),
# para que las cuatro listas del mes hablen del mismo periodo. Si no existe,
# se calcula con utilidades_borde.
# ============================================================

# Corte POR ZONA (lo deja el notebook 3 en cortes_por_zona.csv): urbano = último mes
# completo; rural = último trimestre cerrado. Cada cliente se puntúa en el corte de su zona.
from utilidades_borde import leer_cortes_por_zona
RUTA_CORTES_ZONA = PREPROC_DIR / "cortes_por_zona.csv"
if RUTA_CORTES_ZONA.exists():
    CORTES = leer_cortes_por_zona(RUTA_CORTES_ZONA)
else:
    serie_b = pd.read_parquet(RUTA_SERIE, columns=["NIU", "periodo", "consumo_kwh_mensual", "es_rural"], engine="pyarrow")
    serie_b["periodo"] = pd.to_datetime(serie_b["periodo"])
    serie_b = serie_b[serie_b["periodo"] >= serie_b["periodo"].max() - pd.DateOffset(months=18)]
    CORTES = leer_cortes_por_zona(None, serie=serie_b, col_grupo="es_rural")
    del serie_b
CORTE_URBANO = pd.Timestamp(CORTES["URBANO"]).to_period("M").to_timestamp()
CORTE_RURAL = pd.Timestamp(CORTES["RURAL"]).to_period("M").to_timestamp()
FECHA_CORTE = max(CORTE_URBANO, CORTE_RURAL)   # mes de la corrida (etiqueta de archivos); normalmente el urbano
ETIQUETA_CORTE = f"{FECHA_CORTE:%Y-%m}"

# --- Población elegible (automática) ---
en_tc2 = atrib.index.intersection(por_niu_otros.index)
clases_idos = (atrib.loc[en_tc2, "clase_servicio"].dropna())
clases_idos = clases_idos[~clases_idos.isin(CLASES_EXCLUIDAS_SIEMPRE) & clases_idos.str.match(r"^[A-Z]{2}$")]
if HAY_ARCHIVO_OTROS and len(clases_idos):
    CLASES_ELEGIBLES = set(clases_idos.unique()) | {"IR"}
    origen_clases = "clases observadas entre los que se fueron (archivo de otros comercializadores) + IR"
else:
    CLASES_ELEGIBLES = set(CLASES_POR_DEFECTO)
    origen_clases = "por defecto (no hay archivo de otros comercializadores con historia TC2)"
CLASES_ELEGIBLES -= CLASES_EXCLUIDAS_SIEMPRE

elegible = (atrib["clase_servicio"].isin(CLASES_ELEGIBLES)
            & ~atrib["ciclo"].isin(CICLOS_INTERNOS)
            & ~atrib["ciclo"].isin(CICLOS_SIN_GESTION)          # autogeneradores (ciclo 50): sin gestión
            & ~atrib["ciclo"].eq(CICLO_OTROS_COMERCIALIZADORES))
NIUS_POBLACION = atrib.index[elegible]

print("\nPOBLACIÓN A LA QUE SE LE CALCULA EL RIESGO")
print("-" * 70)
print(f"Clases elegibles : {sorted(CLASES_ELEGIBLES)}  ({origen_clases})")
print(f"Se excluyen      : clases {sorted(CLASES_EXCLUIDAS_SIEMPRE)} y ciclos internos {sorted(CICLOS_INTERNOS)} y sin gestión {sorted(CICLOS_SIN_GESTION)}")
print(f"NIU elegibles    : {len(NIUS_POBLACION):,} de {len(atrib):,}")
display(atrib.loc[NIUS_POBLACION, "clase_servicio"].value_counts().rename("n_NIU").to_frame().T)

# --- Serie mensual (03) solo de la población, como matriz NIU x mes ---
cols_serie = ["NIU", "periodo", "consumo_kwh_mensual"] + (["es_rural"] if "es_rural" in pq.ParquetFile(RUTA_SERIE).schema_arrow.names else [])
serie = pd.read_parquet(RUTA_SERIE, columns=cols_serie, engine="pyarrow")
serie["NIU"] = normalizar_niu(serie["NIU"])
serie = serie[serie["NIU"].isin(NIUS_POBLACION)]
serie["periodo"] = pd.to_datetime(serie["periodo"])
serie = serie[serie["periodo"] <= FECHA_CORTE]
MESES = pd.date_range(serie["periodo"].min(), FECHA_CORTE, freq="MS")
M = serie.pivot_table(index="NIU", columns="periodo", values="consumo_kwh_mensual", aggfunc="sum")
M = M.reindex(columns=MESES)
NIUS = M.index
es_rural_pop = (serie.drop_duplicates("NIU").set_index("NIU")["es_rural"].fillna(False).astype(bool).reindex(NIUS).fillna(False).to_numpy()
                if "es_rural" in serie.columns else np.zeros(len(NIUS), dtype=bool))
# Meses rurales posteriores a su corte: provisionales -> se tratan como sin dato
T_URBANO = int(min(np.searchsorted(MESES, CORTE_URBANO), len(MESES) - 1))
T_RURAL = int(min(np.searchsorted(MESES, CORTE_RURAL), len(MESES) - 1))
M.iloc[np.flatnonzero(es_rural_pop), T_RURAL + 1:] = np.nan
M.iloc[np.flatnonzero(~es_rural_pop), T_URBANO + 1:] = np.nan
X_M = M.to_numpy(dtype="float64")          # NaN = sin fila ese mes
PRESENTE = ~np.isnan(X_M)
print(f"\nMatriz de consumo: {X_M.shape[0]:,} NIU x {X_M.shape[1]} meses ({MESES[0]:%Y-%m} -> {MESES[-1]:%Y-%m})")
print(f"Urbanos {int((~es_rural_pop).sum()):,} (corte {CORTE_URBANO:%Y-%m}) | rurales {int(es_rural_pop.sum()):,} (corte {CORTE_RURAL:%Y-%m})")
del serie
_ = gc.collect()

# --- Atributos comerciales MES A MES de la población (tarifa, promedio semestral, tipo de lectura
#     y de medidor), leídos del histórico y arrastrados hacia adelante. En cada corte se usa el
#     último valor conocido HASTA ese corte, nunca el de la última fila del cliente: para un
#     cliente que ya se fue, esa última fila está después de la salida y contaminaría el modelo.
COLS_MES = ["NIU", "periodo", "tarifa_aplicada_kwh", "consumo_promedio_semestral_kwh", "tipo_lectura", "tipo_medidor"]
partes = []
for ruta in archivos_hist:
    esquema = pq.ParquetFile(ruta).schema_arrow.names
    cols = [c for c in COLS_MES if c in esquema]
    h = pd.read_parquet(ruta, columns=cols, engine="pyarrow")
    h["NIU"] = normalizar_niu(h["NIU"])
    h = h[h["NIU"].isin(NIUS)]
    partes.append(h)
atrib_mes = pd.concat(partes, ignore_index=True)
atrib_mes["periodo"] = pd.to_datetime(atrib_mes["periodo"]).dt.to_period("M").dt.to_timestamp()
atrib_mes = atrib_mes[atrib_mes["periodo"] <= FECHA_CORTE]
for c in COLS_MES[2:]:
    if c not in atrib_mes.columns:
        atrib_mes[c] = np.nan
    atrib_mes[c] = pd.to_numeric(atrib_mes[c], errors="coerce")
atrib_mes.loc[atrib_mes["tarifa_aplicada_kwh"] <= 0, "tarifa_aplicada_kwh"] = np.nan   # tarifa 0 = sin dato
atrib_mes = atrib_mes.sort_values(["NIU", "periodo"]).groupby(["NIU", "periodo"], as_index=False).last()

def matriz_mes(col):
    m = atrib_mes.pivot(index="NIU", columns="periodo", values=col).reindex(index=NIUS, columns=MESES)
    return m.ffill(axis=1).to_numpy(dtype="float64")

TAR_M = matriz_mes("tarifa_aplicada_kwh")
SEM_M = matriz_mes("consumo_promedio_semestral_kwh")
LEC_M = matriz_mes("tipo_lectura")
MED_M = matriz_mes("tipo_medidor")
print(f"Atributos mes a mes: tarifa con dato en el corte para {np.isfinite(TAR_M[:, -1]).sum():,} de {len(NIUS):,} NIU")
del partes, atrib_mes
_ = gc.collect()


## Mes de salida de cada cliente que se fue

Para cada NIU del archivo de otros comercializadores con historia TC2 se busca el mes en que
dejó de ser cliente de EBSA. En los datos reales la huella es clara: el cliente sigue apareciendo
en TC2 pero con **0 kWh** (mediana de consumo después del cambio: 0) y solo meses después
desaparece del archivo.

In [ ]:
# ============================================================
# 5. ETIQUETA: MES DE SALIDA POR NIU
# ============================================================

def inicio_cero_sostenido(fila, presente, min_racha):
    """Primer mes de la racha final de ceros (>= min_racha meses, hasta la última fila del NIU)."""
    idx = np.where(presente)[0]
    if len(idx) == 0:
        return None
    ult = idx[-1]
    k = ult
    while k >= 0 and presente[k] and fila[k] <= 0:
        k -= 1
    largo = ult - k
    return k + 1 if largo >= min_racha else None

etq = pd.DataFrame(index=atrib.index)
etq["en_archivo_otros"] = etq.index.isin(por_niu_otros.index)
etq["primer_mes_otro"] = por_niu_otros["primer_mes_otro"].reindex(etq.index)
etq["ultimo_mes_otro"] = por_niu_otros["ultimo_mes_otro"].reindex(etq.index)
etq["primer_mes_ciclo97"] = atrib["primer_mes_ciclo97"]
etq["ultimo_mes_tc2"] = atrib["ultimo_mes_tc2"]

# consumo cero sostenido solo se calcula en la población (matriz)
cero_ini = {}
for i, niu in enumerate(NIUS):
    k = inicio_cero_sostenido(X_M[i], PRESENTE[i], MESES_CERO_SOSTENIDO)
    if k is not None:
        cero_ini[niu] = MESES[k]
etq["inicio_cero_sostenido"] = pd.Series(cero_ini, dtype="datetime64[ns]").reindex(etq.index)

positivo = etq["en_archivo_otros"] | etq["primer_mes_ciclo97"].notna()
candidatos = pd.concat([
    etq["primer_mes_otro"],
    etq["primer_mes_ciclo97"],
    etq["inicio_cero_sostenido"].where(positivo),
    (etq["ultimo_mes_tc2"] + pd.DateOffset(months=1)).where(positivo & (etq["ultimo_mes_tc2"] < etq["primer_mes_otro"].fillna(pd.Timestamp.max))),
], axis=1)
etq["mes_salida"] = candidatos.min(axis=1).where(positivo)
etq["mes_salida"] = pd.to_datetime(etq["mes_salida"]).dt.to_period("M").dt.to_timestamp()
etq["fuente_salida"] = np.select(
    [etq["primer_mes_ciclo97"].notna() & etq["mes_salida"].eq(etq["primer_mes_ciclo97"]),
     etq["inicio_cero_sostenido"].notna() & etq["mes_salida"].eq(etq["inicio_cero_sostenido"]),
     etq["mes_salida"].notna() & etq["mes_salida"].eq(etq["primer_mes_otro"]),
     etq["mes_salida"].notna()],
    ["ciclo 97 en TC2", "consumo cero sostenido en TC2", "primer mes en archivo de otros", "dejó de aparecer en TC2"],
    default="")

# Estado actual de los que están en el archivo de otros
ult_idx = len(MESES) - 1
def consumo_reciente_positivo(niu, n=2):
    if niu not in M.index:
        return False
    v = M.loc[niu].to_numpy()[-n:]
    return bool(np.nansum(v) > 0) and not np.isnan(v).all()
regreso = pd.Series({niu: consumo_reciente_positivo(niu) for niu in etq.index[etq["en_archivo_otros"]]})
etq["regreso_a_ebsa"] = regreso.reindex(etq.index).fillna(False).astype(bool) & (
    etq["ultimo_mes_otro"] < FECHA_CORTE) & (etq["ultimo_mes_otro"] < ULTIMO_MES_ARCHIVO_OTROS if HAY_ARCHIVO_OTROS else False)

n_pos = int(positivo.sum())
print("MES DE SALIDA (clientes del archivo de otros comercializadores con historia TC2, o ciclo 97)")
print("-" * 78)
print(f"NIU en archivo de otros: {len(por_niu_otros):,} | con historia TC2: {int(etq['en_archivo_otros'].sum()):,} | "
      f"por ciclo 97: {int(etq['primer_mes_ciclo97'].notna().sum()):,}")
if n_pos:
    display(etq.loc[positivo, "fuente_salida"].value_counts().rename("n_NIU").to_frame())
    display(etq.loc[positivo, "mes_salida"].dt.strftime("%Y-%m").value_counts().sort_index().rename("n_NIU").to_frame().T)
    print(f"Regresaron a EBSA (consumo reciente > 0 tras dejar el archivo de otros): {int(etq['regreso_a_ebsa'].sum()):,}")
etq.to_csv(RUTA_ETIQUETAS, encoding="utf-8-sig")


## Variables por cliente en una fecha de corte

Las mismas variables se calculan en cada corte histórico (para entrenar y evaluar) y en el
corte actual (para puntuar). Solo miran hacia atrás desde el corte.

In [ ]:
# ============================================================
# 6. VARIABLES EN UN CORTE
# ============================================================

MESES_IDX = {m: i for i, m in enumerate(MESES)}
clase_pop = atrib.loc[NIUS, "clase_servicio"].astype("object").fillna("SIN_DATO")
ciclo_pop = atrib.loc[NIUS, "ciclo"].fillna(-1).astype(int)
estrato_pop = atrib.loc[NIUS, "estrato"].fillna(0)
primer_pop = atrib.loc[NIUS, "primer_mes_tc2"]

# Densidad de salidas por ciclo/zona: cuántos usuarios del archivo de otros (con o sin
# historia TC2) hay por zona, relativo a la población elegible de esa zona.
if HAY_ARCHIVO_OTROS:
    zona_otros = por_niu_otros["zona_archivo"].dropna().astype(int).value_counts()
else:
    zona_otros = pd.Series(dtype="int64")
pob_por_ciclo = ciclo_pop.value_counts()
tasa_zona = (zona_otros.reindex(pob_por_ciclo.index).fillna(0) / pob_por_ciclo).fillna(0)
tasa_zona_pop = ciclo_pop.map(tasa_zona).fillna(0)

CATEGORICAS = ["clase_servicio", "ciclo"]

def variables_en_corte(t_idx):
    """DataFrame de variables para todos los NIU de la población en el corte MESES[t_idx]."""
    v12 = X_M[:, max(0, t_idx - 11): t_idx + 1]
    v6 = X_M[:, max(0, t_idx - 5): t_idx + 1]
    v3 = X_M[:, max(0, t_idx - 2): t_idx + 1]
    with np.errstate(all="ignore"):
        meses_validos_12 = np.sum(~np.isnan(v12), axis=1)
        media_12 = np.nanmean(v12, axis=1)
        mediana_12 = np.nanmedian(v12, axis=1)
        std_12 = np.nanstd(v12, axis=1)
        max_12 = np.nanmax(v12, axis=1)
        media_6 = np.nanmean(v6, axis=1)
        media_3 = np.nanmean(v3, axis=1)
        min_3 = np.nanmin(v3, axis=1)
        actual = X_M[:, t_idx]
        ceros_12 = np.sum(np.nan_to_num(v12, nan=1.0) <= 0, axis=1)
        ceros_3 = np.sum(np.nan_to_num(v3, nan=1.0) <= 0, axis=1)
        presentes_3 = np.sum(~np.isnan(v3), axis=1)
        # pendiente de los últimos 6 meses (kWh por mes) relativa a la media
        n6 = v6.shape[1]
        xs = np.arange(n6) - (n6 - 1) / 2
        v6f = np.where(np.isnan(v6), np.nanmean(v6, axis=1, keepdims=True), v6)
        pendiente_6 = np.nansum((v6f - np.nanmean(v6f, axis=1, keepdims=True)) * xs, axis=1) / np.sum(xs ** 2)
        pendiente_6_rel = np.where(media_6 > 0, pendiente_6 / media_6, 0.0)
        cv_12 = np.where(media_12 > 0, std_12 / media_12, 0.0)
        ratio_3_12 = np.where(media_12 > 0, media_3 / media_12, np.nan)
        ratio_1_6 = np.where(media_6 > 0, actual / media_6, np.nan)
        tarifa_t = TAR_M[:, t_idx]                                    # última tarifa real conocida hasta el corte
        mediana_tarifa_t = np.nanmedian(tarifa_t) if np.isfinite(tarifa_t).any() else np.nan
        tarifa_rel = tarifa_t / mediana_tarifa_t                      # relativa a la población en ese corte
        sem_t = SEM_M[:, t_idx]
        ratio_sem = np.where(media_12 > 0, sem_t / media_12, np.nan)
        lectura_t = np.nan_to_num(LEC_M[:, t_idx], nan=0.0)
        medidor_t = np.nan_to_num(MED_M[:, t_idx], nan=0.0)
    meses_historia = ((MESES[t_idx].year - primer_pop.dt.year) * 12 + (MESES[t_idx].month - primer_pop.dt.month)).to_numpy()
    df = pd.DataFrame({
        "NIU": NIUS, "fecha_corte": MESES[t_idx],
        "consumo_actual_kwh": actual, "consumo_prom_3m_kwh": media_3, "consumo_prom_6m_kwh": media_6,
        "consumo_prom_12m_kwh": media_12, "mediana_12m_kwh": mediana_12, "max_12m_kwh": max_12, "min_3m_kwh": min_3,
        "cv_12m": cv_12, "ratio_3m_vs_12m": ratio_3_12, "ratio_actual_vs_6m": ratio_1_6,
        "pendiente_6m_rel": pendiente_6_rel, "meses_validos_12m": meses_validos_12, "meses_cero_12m": ceros_12,
        "meses_cero_3m": ceros_3, "presentes_3m": presentes_3, "meses_historia": meses_historia,
        "log_consumo_12m": np.log1p(np.nan_to_num(media_12, nan=0.0)),
        "tarifa_kwh": tarifa_t, "tarifa_rel": tarifa_rel, "estrato": estrato_pop.to_numpy(),
        "tipo_medidor": medidor_t, "tipo_lectura": lectura_t, "consumo_promedio_semestral_kwh": sem_t,
        "ratio_prom_semestral": ratio_sem, "tasa_salida_zona": tasa_zona_pop.to_numpy(),
        "clase_servicio": clase_pop.to_numpy(), "ciclo": ciclo_pop.to_numpy(),
    })
    return df

VARIABLES_NUM = ["consumo_actual_kwh", "consumo_prom_3m_kwh", "consumo_prom_6m_kwh", "consumo_prom_12m_kwh",
                 "mediana_12m_kwh", "max_12m_kwh", "min_3m_kwh", "cv_12m", "ratio_3m_vs_12m", "ratio_actual_vs_6m",
                 "pendiente_6m_rel", "meses_validos_12m", "meses_cero_12m", "meses_cero_3m", "presentes_3m",
                 "meses_historia", "log_consumo_12m", "tarifa_rel", "estrato", "tipo_medidor", "tipo_lectura",
                 "ratio_prom_semestral", "tasa_salida_zona"]
VARIABLES = VARIABLES_NUM + CATEGORICAS

def preparar_X(df):
    X = df[VARIABLES].copy()
    for c in CATEGORICAS:
        X[c] = X[c].astype("category")
    return X

def es_activo(df):
    """Cliente que se puede puntuar en su corte: presente en los últimos 3 meses, con historia
    suficiente y consumo en el último año (los que ya están en cero sostenido no son 'riesgo', ya se fueron)."""
    return (df["presentes_3m"] >= 1) & (df["meses_validos_12m"] >= MIN_MESES_HISTORIA) & (df["consumo_prom_12m_kwh"] > 0)

mes_salida_pop = etq["mes_salida"].reindex(NIUS)
print("Variables definidas:", len(VARIABLES))


## Entrenamiento y evaluación por cortes en el tiempo (modo reentrenar)

Cada corte histórico `t` aporta una fila por cliente activo con la respuesta
"salió en (t, t+6]". Se evalúa en los últimos cortes con ventana completa entrenando solo
con cortes cuya ventana termina antes: el modelo nunca ve el futuro que se le pide predecir.
Sobre esa misma validación, Optuna (semilla fija) busca la regularización de LightGBM que mejor ordena
a los que se fueron; el primer ensayo es siempre la configuración base, así que nunca se elige algo peor.
Como los parámetros se eligen mirando esa validación, su fila de métricas es un poco optimista; la fila
"parámetros base" no lo es, y la prueba definitiva es el seguimiento mes a mes de los cortes reales.

In [ ]:
# ============================================================
# 7. CONJUNTO DE ENTRENAMIENTO
# ============================================================
rng = np.random.default_rng(SEED)
salida_arr = mes_salida_pop.to_numpy()

def etiquetas_en_corte(t_idx):
    t = MESES[t_idx]
    fin = t + pd.DateOffset(months=HORIZONTE_MESES)
    y = (mes_salida_pop > t) & (mes_salida_pop <= fin)
    ya_salio = mes_salida_pop <= t
    return y.to_numpy(), ya_salio.to_numpy()

def construir_filas(idx_cortes, tasa_neg=1.0, con_etiqueta=True):
    partes = []
    for t_idx in idx_cortes:
        df = variables_en_corte(t_idx)
        activo = es_activo(df).to_numpy()
        if con_etiqueta:
            y, ya_salio = etiquetas_en_corte(t_idx)
            usable = activo & ~ya_salio
            df["y"] = y.astype(int)
            df["peso"] = 1.0
            keep = usable & (y | (rng.random(len(df)) < tasa_neg))
            df.loc[~y & keep, "peso"] = 1.0 / tasa_neg
            partes.append(df[keep])
        else:
            partes.append(df[activo])
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

t_actual = len(MESES) - 1
idx_completos = [i for i in range(12, t_actual + 1) if i + HORIZONTE_MESES <= t_actual]   # ventana observada completa
metricas = []
importancia = None
modelo = None
info_modelo = {}

if MODO == "reentrenar":
    n_pob = len(NIUS)
    tasa_neg = min(1.0, MAX_FILAS_NEGATIVAS / max(1, n_pob * len(idx_completos)))
    print(f"Cortes con ventana completa: {len(idx_completos)} ({MESES[idx_completos[0]]:%Y-%m} -> {MESES[idx_completos[-1]]:%Y-%m})")
    print(f"Muestreo de negativos: {tasa_neg:.1%} (los positivos van completos, con peso 1)")
    datos = construir_filas(idx_completos, tasa_neg=tasa_neg)
    n_pos_total = int(datos["y"].sum())
    n_niu_pos = datos.loc[datos["y"].eq(1), "NIU"].nunique()
    print(f"Filas: {len(datos):,} | positivas: {n_pos_total:,} (de {n_niu_pos:,} NIU distintos)")
    TASA_BASE = float(np.average(datos["y"], weights=datos["peso"]))
    print(f"Tasa base (salidas en 6 meses por cliente activo): {TASA_BASE:.3%}")
else:
    datos = None


In [ ]:
# ============================================================
# 8. MODELO: LIGHTGBM CON EVALUACIÓN TEMPORAL (o SIMILITUD si hay pocos ejemplos)
# ============================================================
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score

try:
    import lightgbm as lgb
    HAY_LGB = True
except ImportError:
    HAY_LGB = False
    from sklearn.ensemble import HistGradientBoostingClassifier

# Parámetros de partida (los de la primera versión). En modo reentrenar Optuna busca
# alrededor de ellos y se queda con los que mejor ordenan en la validación temporal.
PARAMS_BASE = {"n_estimators": 400, "learning_rate": 0.03, "num_leaves": 15, "min_child_samples": 40,
               "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 5.0, "reg_alpha": 0.0, "min_split_gain": 0.0}

def entrenar(X, y, w, params=None):
    """Los positivos son muy pocos, así que se les da más peso (scale_pos_weight). Eso mejora el
    ranking pero infla la probabilidad; predecir() la devuelve a la escala real."""
    params = dict(PARAMS_BASE if params is None else params)
    pos = max(1.0, float(np.sum(w[y == 1]))); neg = max(1.0, float(np.sum(w[y == 0])))
    peso_pos = min(50.0, neg / pos)
    if HAY_LGB:
        m = lgb.LGBMClassifier(**params, subsample_freq=1, scale_pos_weight=peso_pos, random_state=SEED, verbose=-1)
        m.fit(X, y, sample_weight=w, categorical_feature=CATEGORICAS)
    else:
        Xn = X.copy()
        for c in CATEGORICAS:
            Xn[c] = Xn[c].cat.codes
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, random_state=SEED)
        w2 = np.where(y == 1, w * peso_pos, w)
        m.fit(Xn, y, sample_weight=w2)
    m.peso_positivos_ = peso_pos
    return m

def predecir(m, X):
    if HAY_LGB:
        p = m.predict_proba(X)[:, 1]
    else:
        Xn = X.copy()
        for c in CATEGORICAS:
            Xn[c] = Xn[c].cat.codes
        p = m.predict_proba(Xn)[:, 1]
    k = getattr(m, "peso_positivos_", 1.0)
    return p / (p + (1.0 - p) * k)      # deshacer el peso: probabilidad en la escala real

def puntaje_similitud(df):
    """Sin ejemplos suficientes: parecido al perfil de los que se fueron (tamaño, clase, zona, caída reciente)."""
    s = np.zeros(len(df))
    s += np.clip(df["log_consumo_12m"].to_numpy() / 12.0, 0, 1) * 0.4
    s += np.clip(df["tasa_salida_zona"].to_numpy() * 20, 0, 1) * 0.2
    s += np.clip(1 - df["ratio_3m_vs_12m"].fillna(1).to_numpy(), 0, 1) * 0.3
    s += np.clip(df["meses_cero_3m"].to_numpy() / 3, 0, 1) * 0.1
    return s

def metricas_ranking(y, p, w, etiqueta):
    y = np.asarray(y); p = np.asarray(p); w = np.asarray(w)
    out = {"conjunto": etiqueta, "n_filas": len(y), "n_positivos": int(y.sum())}
    if y.sum() > 0 and y.sum() < len(y):
        out["AUC"] = round(roc_auc_score(y, p, sample_weight=w), 4)
        out["precision_promedio"] = round(average_precision_score(y, p, sample_weight=w), 4)
        orden = np.argsort(-p)
        base = y.sum() / len(y)
        for k in [50, 100, 200, 500]:
            if k <= len(y):
                top = y[orden[:k]]
                out[f"precision_top{k}_pct"] = round(top.mean() * 100, 2)
                out[f"recall_top{k}_pct"] = round(top.sum() / y.sum() * 100, 2)
                out[f"lift_top{k}"] = round(top.mean() / base, 2) if base > 0 else np.nan
    return out

if MODO == "reentrenar":
    if n_niu_pos >= MIN_POSITIVOS_MODELO:
        METODO = "LightGBM" if HAY_LGB else "HistGradientBoosting"
        # --- evaluación temporal: validar en los últimos cortes, entrenar con los que no se solapan ---
        cortes_val = idx_completos[-CORTES_VALIDACION:] if len(idx_completos) > CORTES_VALIDACION + 6 else idx_completos[-max(1, len(idx_completos)//3):]
        t_val_min = min(cortes_val)
        cortes_train = [i for i in idx_completos if i + HORIZONTE_MESES <= t_val_min]
        tr = datos[datos["fecha_corte"].isin(MESES[cortes_train])]
        va = datos[datos["fecha_corte"].isin(MESES[cortes_val])]
        print(f"Entrenamiento: cortes {MESES[cortes_train[0]]:%Y-%m} -> {MESES[cortes_train[-1]]:%Y-%m} "
              f"({len(tr):,} filas, {int(tr['y'].sum())} positivas)")
        print(f"Validación   : cortes {MESES[cortes_val[0]]:%Y-%m} -> {MESES[cortes_val[-1]]:%Y-%m} "
              f"({len(va):,} filas, {int(va['y'].sum())} positivas)")
        PARAMS_FINAL = dict(PARAMS_BASE)
        ensayos = None
        if tr["y"].sum() >= 5 and va["y"].sum() >= 1:
            X_tr, y_tr, w_tr = preparar_X(tr), tr["y"].to_numpy(), tr["peso"].to_numpy()
            X_va, y_va, w_va = preparar_X(va), va["y"].to_numpy(), va["peso"].to_numpy()

            # --- Optuna: buscar la regularización que mejor ordena en la validación temporal ---
            # Objetivo: precisión promedio (área bajo precisión-recall), que mira la parte alta del
            # ranking y es más estable que la precisión en el top 100 cuando hay 10-30 salidas por corte.
            # El primer ensayo es la configuración base, así el resultado nunca es peor que ella.
            try:
                import optuna
                optuna.logging.set_verbosity(optuna.logging.WARNING)
                HAY_OPTUNA = HAY_LGB
            except ImportError:
                HAY_OPTUNA = False
                print("⚠ optuna no está instalado: se usan los parámetros base (pip install optuna).")

            if HAY_OPTUNA and N_ENSAYOS_OPTUNA > 0:
                def objetivo(trial):
                    params = {
                        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
                        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                        "num_leaves": trial.suggest_int("num_leaves", 4, 31),
                        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300, step=10),
                        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
                        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 100.0, log=True),
                        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
                        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
                    }
                    m = entrenar(X_tr, y_tr, w_tr, params)
                    p = predecir(m, X_va)
                    trial.set_user_attr("AUC", float(roc_auc_score(y_va, p, sample_weight=w_va)))
                    orden = np.argsort(-p)
                    trial.set_user_attr("precision_top100_pct", float(y_va[orden[:100]].mean() * 100))
                    return float(average_precision_score(y_va, p, sample_weight=w_va))

                estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
                estudio.enqueue_trial(PARAMS_BASE)
                print(f"\nBuscando hiperparámetros con Optuna ({N_ENSAYOS_OPTUNA} ensayos, semilla {SEED})...")
                estudio.optimize(objetivo, n_trials=N_ENSAYOS_OPTUNA, show_progress_bar=False)
                ensayos = estudio.trials_dataframe(attrs=("number", "value", "params", "user_attrs")).sort_values("value", ascending=False)
                ensayos = ensayos.rename(columns={"value": "precision_promedio_validacion"})
                ensayos.to_csv(FUGA_DIR / "optuna_ensayos_fuga.csv", index=False, encoding="utf-8-sig")
                PARAMS_FINAL = {**PARAMS_BASE, **estudio.best_params}
                base_valor = estudio.trials[0].value
                print(f"  configuración base : precisión promedio {base_valor:.4f}")
                print(f"  mejor ensayo (#{estudio.best_trial.number}): precisión promedio {estudio.best_value:.4f}")
                print("  parámetros elegidos:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in PARAMS_FINAL.items()})
                m_base = entrenar(X_tr, y_tr, w_tr, PARAMS_BASE)
                metricas.append(metricas_ranking(y_va, predecir(m_base, X_va), w_va, "validación temporal — parámetros base"))

            m_val = entrenar(X_tr, y_tr, w_tr, PARAMS_FINAL)
            p_va = predecir(m_val, X_va)
            metricas.append(metricas_ranking(va["y"], p_va, va["peso"], "validación temporal (todos los cortes) — parámetros elegidos"))
            metricas.append(metricas_ranking(y_tr, predecir(m_val, X_tr), w_tr, "entrenamiento de la validación (para ver el sobreajuste)"))
            for t in sorted(va["fecha_corte"].unique()):
                sub = va["fecha_corte"].eq(t)
                metricas.append(metricas_ranking(va.loc[sub, "y"], p_va[sub.to_numpy()], va.loc[sub, "peso"], f"validación corte {pd.Timestamp(t):%Y-%m}"))
            # comparación contra la similitud simple
            metricas.append(metricas_ranking(va["y"], puntaje_similitud(va), va["peso"], "referencia: similitud simple"))
        else:
            print("⚠ Muy pocos positivos para evaluar por separado; se entrena con todo y se reporta sin validación.")
        # --- modelo final con todos los cortes de ventana completa, con los parámetros elegidos ---
        modelo = entrenar(preparar_X(datos), datos["y"].to_numpy(), datos["peso"].to_numpy(), PARAMS_FINAL)
        json.dump(PARAMS_FINAL, open(FUGA_DIR / "hiperparametros_fuga.json", "w"), indent=2)
        p_tr = predecir(modelo, preparar_X(datos))
        metricas.append(metricas_ranking(datos["y"], p_tr, datos["peso"], "entrenamiento (optimista, solo referencia)"))
        if HAY_LGB:
            importancia = pd.DataFrame({"variable": VARIABLES, "importancia": modelo.booster_.feature_importance("gain")})
            importancia["importancia_pct"] = (importancia["importancia"] / importancia["importancia"].sum() * 100).round(2)
            importancia = importancia.sort_values("importancia_pct", ascending=False)
    else:
        METODO = "SIMILITUD (pocos ejemplos)"
        modelo = None
        print(f"⚠ Solo {n_niu_pos} NIU con salida conocida (< {MIN_POSITIVOS_MODELO}): no se entrena un modelo.")
        print("  Se usa un puntaje de similitud con el perfil de los que se fueron; cuando el archivo de")
        print("  otros comercializadores acumule más casos, el modelo se entrena solo en la siguiente corrida reentrenar.")
        metricas.append(metricas_ranking(datos["y"], puntaje_similitud(datos), datos["peso"], "similitud simple (entrenamiento)"))

    info_modelo = {
        "metodo": METODO, "fecha_corte_modelo": ETIQUETA_CORTE, "entrenado_en": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "clases_elegibles": sorted(CLASES_ELEGIBLES), "origen_clases": origen_clases, "variables": VARIABLES,
        "tasa_base": TASA_BASE, "n_positivos_niu": int(n_niu_pos), "n_filas_entrenamiento": int(len(datos)),
        "horizonte_meses": HORIZONTE_MESES, "hiperparametros": PARAMS_FINAL if modelo is not None else None,
    }
    joblib.dump({"modelo": modelo, "info": info_modelo}, RUTA_MODELO)
    print("\nModelo guardado:", RUTA_MODELO, "| método:", METODO)
    from utilidades_versiones import guardar_version
    print("Versión guardada:", guardar_version(BASE_DIR, "riesgo_fuga", RUTA_MODELO, FECHA_CORTE,
                                                notas=f"{METODO}; {n_niu_pos} NIU con salida conocida", n_positivos_niu=int(n_niu_pos)))
    VERSION_USADA = "recién entrenado"
    if metricas:
        tabla_m = pd.DataFrame(metricas)
        tabla_m.to_csv(RUTA_METRICAS, index=False, encoding="utf-8-sig")
        print("\nMÉTRICAS (ranking: qué tan arriba quedan los que efectivamente se fueron)")
        print("-" * 78)
        display(tabla_m)
    if importancia is not None:
        importancia.to_csv(RUTA_IMPORTANCIA, index=False, encoding="utf-8-sig")
        print("\nVARIABLES MÁS IMPORTANTES")
        display(importancia.head(12))
else:
    from utilidades_versiones import resolver_modelo
    if not RUTA_MODELO.exists() and not os.environ.get("EBSA_VERSION_MODELO", "").strip():
        raise FileNotFoundError(
            f"No existe {RUTA_MODELO}.\nEn modo aplicar se necesita un modelo guardado: corre primero\n"
            "    python pipeline_mensual.py --modo reentrenar --solo 14")
    RUTA_MODELO_USADA, VERSION_USADA = resolver_modelo(BASE_DIR, "riesgo_fuga", RUTA_MODELO)
    print(f"Modelo de riesgo de fuga: versión {VERSION_USADA} -> {RUTA_MODELO_USADA.name}")
    guardado = joblib.load(RUTA_MODELO_USADA)
    modelo, info_modelo = guardado["modelo"], guardado["info"]
    METODO = info_modelo["metodo"]
    TASA_BASE = info_modelo["tasa_base"]
    if set(info_modelo["variables"]) != set(VARIABLES):
        raise ValueError("Las variables del modelo guardado no coinciden con las de este notebook: reentrenar.")
    if set(info_modelo["clases_elegibles"]) != set(CLASES_ELEGIBLES):
        print(f"⚠ La población elegible cambió (modelo: {info_modelo['clases_elegibles']} | ahora: {sorted(CLASES_ELEGIBLES)}).")
        print("  Se puntúa igual, pero conviene reentrenar para que el modelo vea las clases nuevas.")
    antig = (FECHA_CORTE.year - int(info_modelo["fecha_corte_modelo"][:4])) * 12 + (FECHA_CORTE.month - int(info_modelo["fecha_corte_modelo"][5:7]))
    print(f"Modelo cargado: {METODO} | corte del modelo {info_modelo['fecha_corte_modelo']} | hace {antig} mes(es)")
    if antig > MESES_MAX_SIN_REENTRENAR:
        print(f"⚠ El modelo tiene más de {MESES_MAX_SIN_REENTRENAR} meses: programar una corrida en modo reentrenar.")


## Puntuar el corte actual

In [ ]:
# ============================================================
# 9. PROBABILIDAD DE SALIDA EN EL CORTE ACTUAL
# ============================================================
# Cada cliente se puntúa en el corte de su zona: urbanos en T_URBANO, rurales en T_RURAL
actual_urb = variables_en_corte(T_URBANO)
actual_rur = variables_en_corte(T_RURAL)
actual = pd.concat([actual_urb[~es_rural_pop], actual_rur[es_rural_pop]]).sort_index()
actual["fecha_corte"] = pd.to_datetime(np.where(es_rural_pop, np.datetime64(CORTE_RURAL, "ns"), np.datetime64(CORTE_URBANO, "ns")))
actual["zona"] = np.where(es_rural_pop, "RURAL", "URBANO")
activo = es_activo(actual)
ya_salio = (pd.to_datetime(mes_salida_pop).to_numpy().astype("datetime64[ns]")
            <= actual["fecha_corte"].to_numpy().astype("datetime64[ns]"))
en_otros_ahora = etq["en_archivo_otros"].reindex(NIUS).to_numpy() & ~etq["regreso_a_ebsa"].reindex(NIUS).to_numpy()
puntuable = activo.to_numpy() & ~ya_salio & ~en_otros_ahora

scores = actual[puntuable].copy()
if modelo is not None:
    scores["prob_fuga_6m"] = predecir(modelo, preparar_X(scores))
else:
    scores["prob_fuga_6m"] = puntaje_similitud(scores) * max(TASA_BASE, 1e-4) * 10   # escala orientativa
scores["metodo"] = METODO

orden_prob = scores["prob_fuga_6m"].rank(ascending=False, method="first")
n_alto = max(MIN_TOP_ALTO, int(round(len(scores) * PCT_TOP_ALTO)))
n_medio = max(n_alto, int(round(len(scores) * PCT_TOP_MEDIO)))
umbral_alto = MULT_ALTO * TASA_BASE if TASA_BASE > 0 else np.inf     # sin tasa base solo aplica el top %
umbral_medio = MULT_MEDIO * TASA_BASE if TASA_BASE > 0 else np.inf
scores["nivel_riesgo"] = np.select(
    [(scores["prob_fuga_6m"] >= umbral_alto) | (orden_prob <= n_alto),
     (scores["prob_fuga_6m"] >= umbral_medio) | (orden_prob <= n_medio)],
    ["ALTO", "MEDIO"], default="BAJO")

# Valor: solo con tarifa real del cliente (sin imputar)
scores["tiene_tarifa"] = scores["tarifa_kwh"].notna() & (scores["tarifa_kwh"] > 0)
scores["valor_facturado_mes"] = np.where(scores["tiene_tarifa"], (scores["consumo_prom_6m_kwh"] * scores["tarifa_kwh"]).round(0), np.nan)
scores["valor_facturado_origen"] = np.where(scores["tiene_tarifa"], "consumo promedio 6m × tarifa real", "sin tarifa")
scores["valor_en_riesgo_mes"] = scores["valor_facturado_mes"]
scores["valor_esperado_perdida_mes"] = (scores["prob_fuga_6m"] * scores["valor_en_riesgo_mes"]).round(0)
scores["variacion_3m_vs_12m_pct"] = ((scores["ratio_3m_vs_12m"] - 1) * 100).round(1)

def senales(r):
    s = []
    if pd.notna(r["variacion_3m_vs_12m_pct"]) and r["variacion_3m_vs_12m_pct"] <= -20:
        s.append(f"consumo últimos 3 meses {r['variacion_3m_vs_12m_pct']:.0f}% vs el año")
    if r["meses_cero_3m"] >= 1:
        s.append(f"{int(r['meses_cero_3m'])} mes(es) en cero de los últimos 3")
    if r["tasa_salida_zona"] > 0 and r["tasa_salida_zona"] >= tasa_zona.median():
        s.append("zona con más salidas que el promedio")
    if r["consumo_prom_12m_kwh"] >= UMBRAL_NO_REGULADO_KWH:
        s.append("tamaño de mercado no regulado")
    elif r["consumo_prom_12m_kwh"] >= 5000:
        s.append("cliente grande")
    return "; ".join(s)
scores["senales"] = scores.apply(senales, axis=1)

# Grupo de consumo (mismos umbrales del pronóstico), zona y clase en texto
if RUTA_PERFILES.exists():
    perf = pd.read_parquet(RUTA_PERFILES, columns=["NIU", "perfil"], engine="pyarrow")
    perf["NIU"] = normalizar_niu(perf["NIU"])
    scores = scores.merge(perf.drop_duplicates("NIU"), on="NIU", how="left")
    scores["grupo_consumo"] = grupo_desde_perfil(scores["perfil"])
    falta = scores["grupo_consumo"].isna() | scores["perfil"].isna()
    scores.loc[falta, "grupo_consumo"] = grupo_desde_mediana(scores.loc[falta, "mediana_12m_kwh"], scores.loc[falta, "meses_cero_12m"] / 12, scores.loc[falta, "meses_validos_12m"])
else:
    scores["grupo_consumo"] = grupo_desde_mediana(scores["mediana_12m_kwh"], scores["meses_cero_12m"] / 12, scores["meses_validos_12m"])
scores = enriquecer_glosario(scores)
scores["modo"] = MODO
scores["fecha_corte_modelo"] = info_modelo.get("fecha_corte_modelo", ETIQUETA_CORTE)

print("RIESGO DE FUGA — CORTE", ETIQUETA_CORTE)
print("-" * 78)
print(f"Población elegible: {len(NIUS):,} | puntuados (activos): {len(scores):,} | "
      f"ya con otro comercializador o en cero sostenido: {int((ya_salio | en_otros_ahora).sum()):,}")
print(f"Tasa base del modelo: {TASA_BASE:.2%}  ->  ALTO: prob ≥ {umbral_alto:.2%} o top {n_alto} | "
      f"MEDIO: prob ≥ {umbral_medio:.2%} o top {n_medio}")
display(scores.groupby("nivel_riesgo").agg(n_clientes=("NIU", "size"), valor_en_riesgo_mes=("valor_en_riesgo_mes", "sum"),
                                            valor_esperado_perdida_mes=("valor_esperado_perdida_mes", "sum")).reindex(["ALTO", "MEDIO", "BAJO"]))


## Historial, listas y seguimiento

In [ ]:
# ============================================================
# 10. RECURRENCIA CONTRA LOS CORTES ANTERIORES Y LISTAS
# ============================================================
patron_h = re.compile(r"^riesgo_fuga_corte_(\d{4}-\d{2})\.csv$")
listas_previas = {}
for ruta in sorted(HISTORIAL_DIR.glob("riesgo_fuga_corte_*.csv")):
    m = patron_h.match(ruta.name)
    if not m:
        continue
    corte = pd.Timestamp(m.group(1) + "-01")
    if corte >= FECHA_CORTE:
        continue
    prev = pd.read_csv(ruta, usecols=["NIU", "nivel_riesgo"], dtype={"NIU": "string"})
    listas_previas[corte] = set(prev.loc[prev["nivel_riesgo"].isin(["ALTO", "MEDIO"]), "NIU"].str.strip())

def meses_entre(a, b):
    return (b.year - a.year) * 12 + (b.month - a.month)

senalado = scores["nivel_riesgo"].isin(["ALTO", "MEDIO"])
consec = pd.Series(np.where(senalado, 1, 0), index=scores["NIU"].astype("string"))
for k in range(1, 25):
    ck = FECHA_CORTE - pd.DateOffset(months=k)
    if ck not in listas_previas:
        break
    viva = consec.eq(k)
    consec[viva & consec.index.isin(listas_previas[ck])] = k + 1
veces = pd.Series(0, index=consec.index)
for c, nius in listas_previas.items():
    if 1 <= meses_entre(c, FECHA_CORTE) <= 12:
        veces[veces.index.isin(nius)] += 1
scores["meses_consecutivos_en_lista"] = consec.to_numpy()
scores["veces_en_lista_12m"] = veces.to_numpy()
scores["estado_en_lista"] = np.where(~senalado, "", np.select(
    [scores["meses_consecutivos_en_lista"] >= 2, scores["veces_en_lista_12m"] >= 1], ["PERSISTENTE", "REINCIDENTE"], "NUEVO"))

scores = scores.sort_values(["valor_esperado_perdida_mes", "prob_fuga_6m"], ascending=[False, False], na_position="last").reset_index(drop=True)
scores["ranking"] = np.arange(1, len(scores) + 1)
senalado = scores["nivel_riesgo"].isin(["ALTO", "MEDIO"])

COLS_SALIDA = [
    "ranking", "NIU", "nivel_riesgo", "prob_fuga_6m", "valor_esperado_perdida_mes", "valor_en_riesgo_mes",
    "ciclo", "zona", "zona_nombre", "clase_servicio", "clase_servicio_nombre", "estrato", "grupo_consumo",
    "consumo_actual_kwh", "consumo_prom_3m_kwh", "consumo_prom_6m_kwh", "consumo_prom_12m_kwh",
    "variacion_3m_vs_12m_pct", "meses_cero_3m", "meses_cero_12m", "consumo_promedio_semestral_kwh",
    "tarifa_kwh", "tiene_tarifa", "valor_facturado_mes", "valor_facturado_origen",
    "tipo_medidor_nombre", "tipo_lectura_nombre", "senales",
    "estado_en_lista", "meses_consecutivos_en_lista", "veces_en_lista_12m",
    "metodo", "fecha_corte", "fecha_corte_modelo", "modo",
]
COLS_SALIDA = [c for c in COLS_SALIDA if c in scores.columns]
for c in ["consumo_actual_kwh", "consumo_prom_3m_kwh", "consumo_prom_6m_kwh", "consumo_prom_12m_kwh", "consumo_promedio_semestral_kwh"]:
    scores[c] = scores[c].round(1)
scores["prob_fuga_6m"] = scores["prob_fuga_6m"].round(4)

scores[COLS_SALIDA].to_csv(RUTA_SCORES, index=False, encoding="utf-8-sig")
scores[COLS_SALIDA].to_csv(HISTORIAL_DIR / f"riesgo_fuga_corte_{ETIQUETA_CORTE}.csv", index=False, encoding="utf-8-sig")

gerencial = scores[senalado][COLS_SALIDA]
gerencial.to_csv(RUTA_GERENCIAL, index=False, encoding="utf-8-sig")
por_zona = scores[senalado].sort_values(["ciclo", "prob_fuga_6m"], ascending=[True, False])
por_zona["orden_en_zona"] = por_zona.groupby("ciclo").cumcount() + 1
por_zona[["orden_en_zona"] + COLS_SALIDA].to_csv(RUTA_POR_ZONA, index=False, encoding="utf-8-sig")

print(f"Lista gerencial (ALTO + MEDIO): {len(gerencial):,} clientes")
print("  estado en la lista:", ", ".join(f"{k} {v:,}" for k, v in gerencial["estado_en_lista"].value_counts().items()))
display(gerencial.head(20))

# Resúmenes por zona y por grupo
res_zona = (scores.groupby(["ciclo", "zona_nombre"]).agg(
    clientes_puntuados=("NIU", "size"), riesgo_alto=("nivel_riesgo", lambda s: (s == "ALTO").sum()),
    riesgo_medio=("nivel_riesgo", lambda s: (s == "MEDIO").sum()),
    valor_esperado_perdida_mes=("valor_esperado_perdida_mes", "sum")).reset_index().sort_values("valor_esperado_perdida_mes", ascending=False))
res_zona.to_csv(RUTA_RESUMEN_ZONA, index=False, encoding="utf-8-sig")
res_grupo = (scores.groupby("grupo_consumo").agg(
    clientes_puntuados=("NIU", "size"), riesgo_alto=("nivel_riesgo", lambda s: (s == "ALTO").sum()),
    riesgo_medio=("nivel_riesgo", lambda s: (s == "MEDIO").sum()),
    prob_media=("prob_fuga_6m", "mean"), valor_esperado_perdida_mes=("valor_esperado_perdida_mes", "sum"))
    .reindex([g for g in GRUPOS_CONSUMO_ORDEN if g in scores["grupo_consumo"].unique()]).reset_index())
res_grupo.to_csv(RUTA_RESUMEN_GRUPO, index=False, encoding="utf-8-sig")
print("\nPOR GRUPO DE CONSUMO"); display(res_grupo)
print("\nPOR ZONA (top 10 por valor esperado)"); display(res_zona.head(10))


In [ ]:
# ============================================================
# 11. CLIENTES QUE YA ESTÁN CON OTRO COMERCIALIZADOR + VIGILANCIA NO REGULADOS + PERFIL DE LOS QUE SE FUERON
# ============================================================
ya = por_niu_otros.copy()
ya.index.name = "NIU"
ya = ya.join(etq[["mes_salida", "fuente_salida", "ultimo_mes_tc2", "regreso_a_ebsa", "primer_mes_ciclo97"]], how="left")
ya = ya.join(atrib[["ciclo", "clase_servicio", "estrato", "primer_mes_tc2", "tarifa_kwh", "consumo_promedio_semestral_kwh"]], how="left")
if len(NIUS):
    # consumo promedio de los 6 meses anteriores a la salida (en TC2)
    def prom_antes(niu):
        if niu not in M.index or pd.isna(etq.at[niu, "mes_salida"]):
            return np.nan
        k = MESES_IDX.get(etq.at[niu, "mes_salida"])
        if k is None:
            k = len(MESES)
        v = M.loc[niu].to_numpy()[max(0, k - 6):k]
        return float(np.nanmean(v)) if len(v) and not np.isnan(v).all() else np.nan
    ya["consumo_prom_6m_antes_salida_kwh"] = [prom_antes(n) for n in ya.index]
else:
    ya["consumo_prom_6m_antes_salida_kwh"] = np.nan
ya["estado"] = np.select(
    [ya["primer_mes_tc2"].isna(), ya["regreso_a_ebsa"].fillna(False).astype(bool),
     ya["ultimo_mes_otro"].notna() & (ya["ultimo_mes_otro"] >= (ULTIMO_MES_ARCHIVO_OTROS if HAY_ARCHIVO_OTROS else pd.Timestamp.max))],
    ["SIN HISTORIA EN TC2 (se fue antes de la historia)", "REGRESÓ A EBSA", "CON OTRO COMERCIALIZADOR"],
    default="SALIÓ DEL ARCHIVO DE OTROS (verificar)")
ya = ya.reset_index()
ya = enriquecer_glosario(ya, consumo_col="consumo_prom_6m_antes_salida_kwh")
ya["valor_facturado_antes_salida_mes"] = np.where(ya["tarifa_kwh"] > 0, (ya["consumo_prom_6m_antes_salida_kwh"] * ya["tarifa_kwh"]).round(0), np.nan)
ya["fecha_corte"] = FECHA_CORTE
COLS_YA = ["NIU", "estado", "comercializador", "municipio", "usuario", "zona_archivo", "nivel_tension", "consumo_prom_otro_kwh",
           "primer_mes_otro", "ultimo_mes_otro", "meses_en_archivo", "mes_salida", "fuente_salida", "primer_mes_tc2", "ultimo_mes_tc2",
           "ciclo", "zona_nombre", "clase_servicio", "clase_servicio_nombre", "estrato", "consumo_prom_6m_antes_salida_kwh",
           "tarifa_kwh", "valor_facturado_antes_salida_mes", "fecha_corte"]
ya = ya.sort_values(["estado", "valor_facturado_antes_salida_mes"], ascending=[True, False])
ya[[c for c in COLS_YA if c in ya.columns]].to_csv(RUTA_YA_FUERA, index=False, encoding="utf-8-sig")
print("CLIENTES YA ATENDIDOS POR OTRO COMERCIALIZADOR")
print("-" * 78)
display(ya["estado"].value_counts().rename("n_NIU").to_frame())
if ya["valor_facturado_antes_salida_mes"].notna().any():
    print(f"Valor que facturaban antes de irse (solo con tarifa real): ${ya['valor_facturado_antes_salida_mes'].sum():,.0f}/mes "
          f"sobre {int(ya['valor_facturado_antes_salida_mes'].notna().sum())} clientes")

# Perfil de los que se fueron (para negocio): por clase, zona y tamaño
if len(ya):
    ya["grupo_tamano"] = pd.cut(ya["consumo_prom_6m_antes_salida_kwh"].fillna(ya["consumo_prom_otro_kwh"]),
                                [-1, 500, 5000, 55000, 1e12], labels=["Pequeño (<500)", "Mediano (500-5.000)", "Grande (5.000-55.000)", "No regulado (≥55.000)"])
    perfil_idos = pd.concat([
        ya.groupby("clase_servicio_nombre", dropna=False).size().rename("n").reset_index().rename(columns={"clase_servicio_nombre": "valor"}).assign(dimension="clase de servicio (TC2)"),
        ya.assign(zona=nombre_zona(ya["zona_archivo"])).groupby("zona", dropna=False).size().rename("n").reset_index().rename(columns={"zona": "valor"}).assign(dimension="zona (archivo)"),
        ya.groupby("grupo_tamano", dropna=False, observed=False).size().rename("n").reset_index().rename(columns={"grupo_tamano": "valor"}).assign(dimension="tamaño"),
        ya.groupby("comercializador", dropna=False).size().rename("n").reset_index().rename(columns={"comercializador": "valor"}).assign(dimension="comercializador"),
        ya.groupby("municipio", dropna=False).size().rename("n").reset_index().rename(columns={"municipio": "valor"}).assign(dimension="municipio"),
    ], ignore_index=True)
    perfil_idos["pct"] = (perfil_idos["n"] / len(ya) * 100).round(1)
    perfil_idos = perfil_idos[["dimension", "valor", "n", "pct"]].sort_values(["dimension", "n"], ascending=[True, False])
    perfil_idos.to_csv(RUTA_PERFIL_IDOS, index=False, encoding="utf-8-sig")

# Vigilancia del mercado no regulado: ciclo 33, clase IR o tamaño ≥ 55 MWh-mes
vig = actual[(actual["ciclo"].eq(CICLO_NO_REGULADOS) | actual["clase_servicio"].eq("IR") | (actual["consumo_prom_12m_kwh"] >= UMBRAL_NO_REGULADO_KWH))].copy()
vig = vig.merge(scores[["NIU", "prob_fuga_6m", "nivel_riesgo", "grupo_consumo"]], on="NIU", how="left")
vig["motivo"] = np.select(
    [vig["ciclo"].eq(CICLO_NO_REGULADOS), vig["clase_servicio"].eq("IR")],
    ["ciclo 33 USUARIOS NO REGULADOS", "clase IR NO REGULADO"], default="consumo ≥ 55.000 kWh/mes (tamaño de no regulado)")
vig = enriquecer_glosario(vig, consumo_col="consumo_prom_6m_kwh")
vig["fecha_corte"] = FECHA_CORTE
vig = vig.sort_values("consumo_prom_12m_kwh", ascending=False)
vig[[c for c in ["NIU", "motivo", "ciclo", "zona_nombre", "clase_servicio", "clase_servicio_nombre", "consumo_actual_kwh", "consumo_prom_6m_kwh",
                 "consumo_prom_12m_kwh", "tarifa_kwh", "valor_facturado_mes", "prob_fuga_6m", "nivel_riesgo", "grupo_consumo", "fecha_corte"] if c in vig.columns]
    ].to_csv(RUTA_VIGILANCIA, index=False, encoding="utf-8-sig")
print(f"\nVigilancia mercado no regulado: {len(vig):,} clientes")
display(vig["motivo"].value_counts().rename("n").to_frame())


In [ ]:
# ============================================================
# 12. SEGUIMIENTO: ¿LOS SEÑALADOS EN CORTES ANTERIORES SE FUERON?
# ============================================================
# Para cada corte anterior con lista guardada: de los clientes ALTO/MEDIO, cuántos tienen
# mes de salida dentro de su ventana de 6 meses (según las etiquetas de hoy, que se
# actualizan cada mes con el archivo de otros comercializadores).
# ============================================================
filas_seg = []
for ruta in sorted(HISTORIAL_DIR.glob("riesgo_fuga_corte_*.csv")):
    m = patron_h.match(ruta.name)
    if not m:
        continue
    corte = pd.Timestamp(m.group(1) + "-01")
    if corte >= FECHA_CORTE:
        continue
    prev = pd.read_csv(ruta, dtype={"NIU": "string"})
    prev = prev[[c for c in ["NIU", "nivel_riesgo", "prob_fuga_6m", "fecha_corte"] if c in prev.columns]]
    prev["NIU"] = normalizar_niu(prev["NIU"])
    prev["mes_salida"] = etq["mes_salida"].reindex(prev["NIU"]).to_numpy()
    corte_fila = (pd.to_datetime(prev["fecha_corte"]).dt.to_period("M").dt.to_timestamp()
                  if "fecha_corte" in prev.columns else pd.Series(corte, index=prev.index))
    fin = corte_fila + pd.DateOffset(months=HORIZONTE_MESES)
    prev["salio_en_ventana"] = (prev["mes_salida"] > corte_fila) & (prev["mes_salida"] <= fin)
    meses_transc = meses_entre(corte, FECHA_CORTE)
    for nivel in ["ALTO", "MEDIO", "BAJO"]:
        sub = prev[prev["nivel_riesgo"].eq(nivel)]
        if len(sub) == 0:
            continue
        filas_seg.append({"fecha_corte_lista": corte, "nivel_riesgo": nivel, "clientes_senalados": len(sub),
                          "salieron_en_ventana": int(sub["salio_en_ventana"].sum()),
                          "pct_salieron": round(sub["salio_en_ventana"].mean() * 100, 2),
                          "meses_transcurridos": meses_transc, "ventana_completa": meses_transc >= HORIZONTE_MESES,
                          "evaluado_en_corte": FECHA_CORTE})
    base = prev["salio_en_ventana"].mean() * 100 if len(prev) else np.nan
    filas_seg.append({"fecha_corte_lista": corte, "nivel_riesgo": "TODOS (tasa base)", "clientes_senalados": len(prev),
                      "salieron_en_ventana": int(prev["salio_en_ventana"].sum()), "pct_salieron": round(base, 2),
                      "meses_transcurridos": meses_transc, "ventana_completa": meses_transc >= HORIZONTE_MESES,
                      "evaluado_en_corte": FECHA_CORTE})
seguimiento = pd.DataFrame(filas_seg)
if len(seguimiento):
    seguimiento.to_csv(RUTA_SEGUIMIENTO, index=False, encoding="utf-8-sig")
    print("SEGUIMIENTO DE LISTAS ANTERIORES")
    print("-" * 78)
    display(seguimiento)
else:
    print("Sin listas de cortes anteriores todavía: el seguimiento empieza en la próxima corrida.")
    if RUTA_SEGUIMIENTO.exists():
        RUTA_SEGUIMIENTO.unlink()


In [ ]:
# ============================================================
# 13. GRÁFICAS Y CIERRE
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
if n_pos:
    etq.loc[positivo, "mes_salida"].dt.to_period("M").value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#eb6834")
    axes[0].set_title("Clientes que se fueron, por mes de salida"); axes[0].set_xlabel(""); axes[0].tick_params(axis="x", labelsize=7)
res_zona.head(12).set_index("zona_nombre")[["riesgo_alto", "riesgo_medio"]].plot(kind="barh", stacked=True, ax=axes[1], color=["#eb6834", "#2a78d6"])
axes[1].set_title("Clientes en riesgo por zona"); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

from utilidades_versiones import registrar_uso
registrar_uso(BASE_DIR, "riesgo_fuga", FECHA_CORTE, VERSION_USADA,
              pd.Timestamp(info_modelo.get("fecha_corte_modelo", ETIQUETA_CORTE) + "-01"), MODO)

print("RIESGO DE FUGA — TERMINADO")
print("=" * 78)
print(f"Corte urbano {CORTE_URBANO:%Y-%m} | corte rural {CORTE_RURAL:%Y-%m} | modo {MODO} | método {METODO}")
print(f"Puntuados: {len(scores):,} | ALTO: {(scores['nivel_riesgo'] == 'ALTO').sum():,} | MEDIO: {(scores['nivel_riesgo'] == 'MEDIO').sum():,}")
print(f"Valor esperado de pérdida (6 meses, por mes): ${scores['valor_esperado_perdida_mes'].sum():,.0f}")
print("\nSalidas:")
for r in [RUTA_SCORES, RUTA_GERENCIAL, RUTA_POR_ZONA, RUTA_YA_FUERA, RUTA_VIGILANCIA, RUTA_PERFIL_IDOS, RUTA_RESUMEN_ZONA,
          RUTA_RESUMEN_GRUPO, RUTA_METRICAS, RUTA_IMPORTANCIA, RUTA_SEGUIMIENTO, RUTA_ETIQUETAS, RUTA_MODELO,
          FUGA_DIR / "hiperparametros_fuga.json", FUGA_DIR / "optuna_ensayos_fuga.csv"]:
    if r.exists():
        print(" •", r)
print(" •", HISTORIAL_DIR / f"riesgo_fuga_corte_{ETIQUETA_CORTE}.csv", "(copia de este corte)")
